# Midterm Manure Q1 Benchmark

Local VS Code notebook for evaluating ChatbotLP on Midterm Problem 1, Question 1: Manure Management.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/Users/mgomezochoa/Documents/ChatbotLP')

In [2]:
import os
os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

from src.midterm_benchmark import (
    DEFAULT_Q1_BENCHMARK_DIR,
    MIDTERM_REASONING_PROMPTS,
    MidtermBenchmarkConfig,
    load_benchmark_files,
    run_midterm_manure_q1_benchmark,
    write_midterm_outputs,
)


## Problem Statement And Reference Solution

In [ ]:
files = load_benchmark_files(DEFAULT_Q1_BENCHMARK_DIR)
reference = files["reference_solution"]

display(Markdown(files["problem_statement"]))

reference_summary = pd.DataFrame([
    {"metric": "objective_value", "value": reference["objective_value"]},
    {"metric": "demand_revenue", "value": reference["demand_revenue"]},
    {"metric": "transport_cost", "value": reference["transport_cost"]},
    {"metric": "supply_cost", "value": reference["supply_cost"]},
])
display(reference_summary)

route_economics = pd.DataFrame([
    {"route": route, "net_value": value}
    for route, value in reference["route_net_values"].items()
])
display(route_economics)
display(reference)


## Canonical And Paraphrased Prompt Runs

In [ ]:
USE_LLM = bool(os.environ.get("GEMINI_API_KEY"))
evaluation_mode = "live_llm" if USE_LLM else "deterministic_fixture"
print(f"evaluation_mode = {evaluation_mode!r}")

config = MidtermBenchmarkConfig(
    prompt_ids=("canonical", "paraphrased"),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    use_deterministic_fixture=not USE_LLM,
    attempt_solve=True,
    run_reasoning=False,
)
report = run_midterm_manure_q1_benchmark(config=config)

display(pd.DataFrame([report["metadata"]]))
display(report["tables"]["case_summary"])
display(report["tables"]["solve_accuracy"])
display(report["tables"]["interpretation_errors"])
display(report["tables"]["interpretation_metadata"])

diagnostic_table_names = [
    "semantic_count_metrics",
    "parameter_multiset_metrics",
    "topology_metrics",
    "technology_yield_metrics",
    "route_economics_metrics",
    "route_association_metrics",
    "transport_link_attribute_metrics",
    "solver_aggregate_metrics",
    "balance_residual_metrics",
    "formulation_completeness_metrics",
    "solve_correctness_metrics",
    "reasoning_readiness_metrics",
    "active_flow_objective_diagnostics",
    "alias_resolution_diagnostics",
]
for table_name in diagnostic_table_names:
    display(Markdown(f"### {table_name}"))
    display(report["tables"].get(table_name, pd.DataFrame()))


In [6]:
canonical_case = report["cases"][0]
display(canonical_case["validation_result"])
display(canonical_case["solution_checks"]["actual_components"])

{'issues': [],
 'missing_parameters': [],
 'invalid_references': [],
 'incomplete_technologies': [],
 'solver_ready': True,
 'benchmark_compatibility': {'Case A': {'compatible': True,
   'explanation': 'no technologies present'},
  'Case B': {'compatible': False, 'explanation': 'no negative bids found'},
  'Case C': {'compatible': False,
   'explanation': 'no technology with both input (neg) and output (pos) yields found'}}}

{'accepted_supply': {'Dairy/EauClaire': 1000.0},
 'accepted_demands': {'Menomonie': 500.0, 'BlackRiverFalls': 500.0},
 'transport_flows': {'EauClaire_to_Menomonie': 500.0,
  'EauClaire_to_BlackRiverFalls': 500.0},
 'demand_revenue': 1000.0,
 'transport_cost': 150.0,
 'supply_cost': 0.0,
 'balance_checks': {'EauClaire': {'supply': 1000.0,
   'total_outgoing_flow': 1000.0,
   'holds': True},
  'Menomonie': {'incoming_flow': 500.0,
   'accepted_demand': 500.0,
   'holds': True},
  'BlackRiverFalls': {'incoming_flow': 500.0,
   'accepted_demand': 500.0,
   'holds': True}},
 'raw_component_rows': [{'component': 'accepted_supply',
   'raw_id': 'S1',
   'raw_bid_id': 'B1',
   'raw_node': 'Eau Claire',
   'raw_product': 'Manure',
   'canonical_resolved_id': 'Dairy/EauClaire',
   'actual_value': 1000.0},
  {'component': 'accepted_demands',
   'raw_id': 'C1',
   'raw_bid_id': 'B2',
   'raw_node': 'Menomonie',
   'raw_product': 'Manure',
   'canonical_resolved_id': 'Menomonie',
   'actual_value':

## Primary Flags


In [ ]:
primary_flags = report["tables"]["case_summary"][[
    "prompt_id",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_attribute_binding_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "failure_type",
]]
display(primary_flags)


## Required Reasoning Prompts

In [ ]:
display(pd.DataFrame(MIDTERM_REASONING_PROMPTS)[["id", "label", "prompt"]])

REASONING_PROMPT_ID = None
reasoning_prompt_ids = (REASONING_PROMPT_ID,) if REASONING_PROMPT_ID else None
reasoning_config = MidtermBenchmarkConfig(
    prompt_ids=("canonical",),
    reasoning_prompt_ids=reasoning_prompt_ids,
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    use_deterministic_fixture=not USE_LLM,
    attempt_solve=True,
    run_reasoning=True,
)
reasoning_report = run_midterm_manure_q1_benchmark(config=reasoning_config)
display(reasoning_report["tables"]["reasoning_prompt_success"])

for row in reasoning_report["cases"][0]["reasoning_results"]:
    display(Markdown(f"### {row['prompt_label']}\n\n{row['response_preview']}"))


## Flow Table

In [ ]:
flow_table = pd.DataFrame([
    {"route": "Eau Claire -> Menomonie", "flow_tons": 500},
    {"route": "Eau Claire -> Black River Falls", "flow_tons": 500},
])
display(flow_table)

## Paper-Style Summary Export

In [ ]:
output_dir = REPO_ROOT / "midterm_outputs"
write_midterm_outputs(report, output_dir)

if "reasoning_report" not in globals():
    reasoning_prompt_ids = (REASONING_PROMPT_ID,) if globals().get("REASONING_PROMPT_ID") else None
    reasoning_config = MidtermBenchmarkConfig(
        prompt_ids=("canonical",),
        reasoning_prompt_ids=reasoning_prompt_ids,
        use_llm=USE_LLM,
        use_llm_for_reasoning=USE_LLM,
        use_deterministic_fixture=not USE_LLM,
        attempt_solve=True,
        run_reasoning=True,
    )
    reasoning_report = run_midterm_manure_q1_benchmark(config=reasoning_config)

reasoning_report["tables"]["reasoning_prompt_success"].to_csv(
    output_dir / "manure_q1_reasoning_prompt_success_canonical.csv",
    index=False,
)
flow_table.to_csv(output_dir / "manure_q1_flow_table.csv", index=False)

paper_summary = report["tables"]["case_summary"][[
    "prompt_id",
    "solver_ready_actual",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_attribute_binding_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "solve_success",
]]
paper_summary.to_csv(output_dir / "manure_q1_paper_style_summary.csv", index=False)
display(paper_summary)
output_dir


## Optional Network Visualization

These cells build directed network graphs from the interpreted `ProblemState` and, when available, the solved `SolverResults`.

Graphviz is optional. If SVG rendering is unavailable on macOS, run:

`brew install graphviz`

`pip install graphviz`


In [ ]:
from IPython.display import SVG

from src.network_graph import build_problem_graph_spec, build_solution_graph_spec
from src.network_visualizer import render_graphviz, render_mermaid

network_output_dir = REPO_ROOT / "midterm_outputs" / "network_graphs"
network_output_dir.mkdir(parents=True, exist_ok=True)


def display_network_graph(spec, name):
    svg_path = network_output_dir / f"{name}.svg"
    try:
        rendered_path = render_graphviz(spec, str(svg_path))
        display(SVG(filename=rendered_path))
        return rendered_path
    except RuntimeError as exc:
        display(Markdown(f"Graphviz unavailable: `{exc}`"))
        display(Markdown("```mermaid\n" + render_mermaid(spec) + "\n```"))
        return None


visual_case = report["cases"][0]
visual_state = visual_case["problem_state"]
problem_spec = build_problem_graph_spec(visual_state)
display_network_graph(problem_spec, "q1_problem_graph")

if visual_case.get("solver_results") is not None and visual_case.get("solve_success"):
    solution_spec = build_solution_graph_spec(visual_state, visual_case["solver_results"])
    display_network_graph(solution_spec, "q1_solution_graph")
else:
    display(Markdown("No successful solver results are available for the solution graph."))
